# Native adapter and barcode removal

This notebook removes terminal Oxford Nanopore ligation adapters and native barcode constructs from `manual_test/ESIB_EQA_2026_SARS1_04.fastq.gz` using Sassy's batched approximate string matcher. It loads every forward and reverse barcode sequence from `adapters.tsv`, so it can remove residual misidentified barcodes from demultiplexed reads and process FASTQs from other native barcode pools.

Sequences are taken from the [Oxford Nanopore Chemistry Technical Document](https://nanoporetech.com/document/chemistry-technical-document#barcode-sequences). Searches use only the explicitly defined left and right construct orientations. The left search prefers the full ligation-adapter top strand and falls back to its 12-base suffix plus both barcode flanks when leading noise or an incomplete adapter is present. A final 32-base forward-barcode-plus-right-flank fallback recovers barcodes beginning within the read; it must start within five bases of the read boundary, allows at most five edits, and is accepted only when it identifies one barcode unambiguously. The right search first includes the corresponding bottom strand after the reverse barcode construct, then falls back to the full 45-base reverse barcode construct for simplex reads. Its final 31-base reverse-flank-plus-barcode fallback recovers noisy barcodes overhanging the read end with the same immediate-boundary, five-edit, unambiguous-barcode constraints. Searches otherwise examine 160-base terminal windows, accept up to 50 bases of terminal noise, and tolerate small basecalling errors or truncated terminal constructs. The original compressed FASTQ remains unchanged; cleaned reads are written to `manual_test/processed/EQA_04.noadapters.fastq`.

In [ ]:
import csv
import gzip
import math
from collections.abc import Iterator
from dataclasses import dataclass
from pathlib import Path
import os
import json

from collections import Counter
from itertools import islice

import sassy
from Bio.SeqIO.QualityIO import FastqGeneralIterator

In [ ]:
DIRECTORY_ROOT = Path(os.getcwd())
INPUT_FASTQ = DIRECTORY_ROOT / "manual_test/ESIB_EQA_2026_SARS1_04.fastq.gz"
OUTPUT_FASTQ = DIRECTORY_ROOT / "manual_test/processed/EQA_04.noadapters.fastq"
BARCODE_TSV = DIRECTORY_ROOT / "adapters.tsv"

In [ ]:
ligation_adapters = json.load(open("ligation_adapters.json"))

LIGATION_ADAPTER_TOP = ligation_adapters["ligation_adapter"]["top_strand"]
LIGATION_ADAPTER_BOTTOM = ligation_adapters["ligation_adapter"]["bottom_strand"]
NATIVE_ADAPTER_TOP = ligation_adapters["native_adapter"]["top_strand"]
NATIVE_ADAPTER_BOTTOM = ligation_adapters["native_adapter"]["bottom_strand"]

In [ ]:
barcode_flanks = json.load(open("barcode_flanks.json"))

FORWARD_FLANK_LEFT = barcode_flanks["forward_flanks"]["left"]
FORWARD_FLANK_RIGHT = barcode_flanks["forward_flanks"]["right"]
REVERSE_FLANK_LEFT = barcode_flanks["reverse_flanks"]["left"]
REVERSE_FLANK_RIGHT = barcode_flanks["reverse_flanks"]["right"]

LEFT_BARCODE_FLANKS = [FORWARD_FLANK_LEFT, FORWARD_FLANK_RIGHT]
RIGHT_BARCODE_FLANKS = [REVERSE_FLANK_LEFT, REVERSE_FLANK_RIGHT]

### Search and removal settings

In [ ]:
LEFT_PARTIAL_ADAPTER_LENGTH = 12
RIGHT_PARTIAL_ADAPTER_LENGTH = 12


LEFT_PARTIAL_BARCODE_LENGTH = 24 + len(FORWARD_FLANK_RIGHT)
RIGHT_PARTIAL_BARCODE_LENGTH = 24 + len(REVERSE_FLANK_LEFT)

In [ ]:
TERMINAL_WINDOW = 160
TERMINAL_SLACK = 50
LEFT_PARTIAL_TERMINAL_SLACK = 5
LEFT_PARTIAL_MAX_EDITS = 5
RIGHT_PARTIAL_MAX_EDITS = 5
RIGHT_PARTIAL_TERMINAL_SLACK = 5
RIGHT_PARTIAL_MAX_EDITS = 5
MAX_ERROR_RATE = 0.10

### base search function

In [ ]:
SEARCHER = sassy.Searcher("iupac", rc=False, alpha=0.5)

### fastq dataclasses

In [ ]:
@dataclass(frozen=True)
class FastqRecord:
    title: str
    sequence: str
    qualities: str


@dataclass(frozen=True)
class TerminalMatch:
    barcode: str
    start: int
    end: int
    cost: int
    cigar: str


@dataclass(frozen=True)
class TrimDecision:
    record: FastqRecord
    left_match: TerminalMatch | None
    right_match: TerminalMatch | None

### Create barcode constructs

In [ ]:
def read_barcode_constructs(path: Path) -> tuple:
    # Read all barcode definitions from the tab-separated file.
    with path.open(newline="") as handle:
        rows = list(csv.DictReader(handle, delimiter="\t"))

    if not rows:
        raise ValueError(f"No barcode records found in {path}")

    # Extract barcode names and normalize all sequences to uppercase.
    barcode_names = [row["barcode"] for row in rows]
    forward_barcodes = [row["Forward sequence"].upper() for row in rows]
    reverse_barcodes = [row["Reverse sequence"].upper() for row in rows]

    # Validate barcode identifiers and sequence lengths.
    if len(set(barcode_names)) != len(barcode_names):
        raise ValueError("Barcode names must be unique")
    if any(len(barcode) != 24 for barcode in forward_barcodes + reverse_barcodes):
        raise ValueError("Every native barcode sequence must be 24 bases")

    # Build the complete left construct:
    # ligation adapter + forward flanks + forward barcode.
    left_constructs = {
        barcode: "".join(
            (LIGATION_ADAPTER_TOP, LEFT_BARCODE_FLANKS[0], forward, LEFT_BARCODE_FLANKS[1])
        )
        for barcode, forward in zip(barcode_names, forward_barcodes)
    }

    # Keep only the adapter suffix and the complete flanked barcode.
    left_partial_adapter_constructs = {
        barcode: construct[
            -(LEFT_PARTIAL_ADAPTER_LENGTH + sum(map(len, LEFT_BARCODE_FLANKS)) + 24):
        ]
        for barcode, construct in left_constructs.items()
    }

    # Build the left-boundary fallback containing the forward barcode
    # and its right flank.
    left_partial_barcode_constructs = {
        barcode: (forward + LEFT_BARCODE_FLANKS[1])[:LEFT_PARTIAL_BARCODE_LENGTH]
        for barcode, forward in zip(barcode_names, forward_barcodes)
    }

    # Build the complete reverse-barcode construct from its two flanks.
    right_barcode_constructs = {
        barcode: "".join(
            (RIGHT_BARCODE_FLANKS[0], reverse, RIGHT_BARCODE_FLANKS[1])
        )
        for barcode, reverse in zip(barcode_names, reverse_barcodes)
    }

    # Keep the prefix used to detect reverse constructs near the right boundary.
    right_partial_barcode_constructs = {
        barcode: construct[:RIGHT_PARTIAL_BARCODE_LENGTH]
        for barcode, construct in right_barcode_constructs.items()
    }

    # Add the ligation-adapter bottom strand for the full right construct.
    right_adapter_constructs = {
        barcode: right_barcode_constructs[barcode] + LIGATION_ADAPTER_BOTTOM
        for barcode in barcode_names
    }

    # Return all construct sets used by the terminal matching logic.
    return (
        left_constructs,
        left_partial_adapter_constructs,
        left_partial_barcode_constructs,
        right_barcode_constructs,
        right_partial_barcode_constructs,
        right_adapter_constructs,
    )


In [ ]:
(
    LEFT_CONSTRUCTS, 
    LEFT_PARTIAL_ADAPTER_CONSTRUCTS,
    LEFT_PARTIAL_BARCODE_CONSTRUCTS,
    RIGHT_BARCODE_CONSTRUCTS,
    RIGHT_PARTIAL_BARCODE_CONSTRUCTS,
    RIGHT_ADAPTER_CONSTRUCTS,
) = read_barcode_constructs(BARCODE_TSV)

### Reusable functions for adapter removal

In [ ]:
def read_fastq(path: Path) -> Iterator[FastqRecord]:
    # Select gzip.open for compressed FASTQ files; otherwise use the regular open.
    opener = gzip.open if path.suffix == ".gz" else open

    # Open the FASTQ as text and yield records one at a time.
    with opener(path, "rt") as handle:
        for title, sequence, qualities in FastqGeneralIterator(handle):
            # Normalize sequences to uppercase while preserving titles and qualities.
            yield FastqRecord(title, sequence.upper(), qualities)


def max_edits(constructs: dict[str, str]) -> int:
    # Collect construct lengths to ensure all batch patterns are equally sized.
    lengths = {len(construct) for construct in constructs.values()}

    # Sassy requires all batch patterns to have the same length.
    if len(lengths) != 1:
        raise ValueError("Sassy batch patterns must have a common length")

    # Allow errors according to the configured maximum error rate.
    return math.ceil(lengths.pop() * MAX_ERROR_RATE)

In [ ]:
def best_terminal_match(
    window: str,
    constructs: dict[str, str],
    terminal: str,
    terminal_slack: int = TERMINAL_SLACK,
    max_allowed_edits: int | None = None,
    require_unique_barcode: bool = False,
) -> TerminalMatch | None:
    # Preserve barcode names so matcher pattern indexes can be mapped back to names.
    barcode_names = tuple(constructs)

    # Encode construct sequences for the sassy searcher.
    patterns = [constructs[barcode].encode("ascii") for barcode in barcode_names]

    # Determine the maximum allowed edit distance for the search.
    cost = max_allowed_edits if max_allowed_edits is not None else max_edits(constructs)
    # Search for all construct patterns within the supplied terminal window.
    matches = SEARCHER.search_many(
        patterns,
        [window.encode("ascii")],
        k=cost,
        threads=1,
        mode="batch_patterns",
    )

    # Keep matches close enough to the requested terminal.
    if terminal == "left":
        matches = [
            match
            for match in matches
            if match.text_start <= terminal_slack
        ]
        placement = lambda match: match.text_start
    elif terminal == "right":
        matches = [
            match
            for match in matches
            if len(window) - match.text_end <= terminal_slack
        ]
        placement = lambda match: len(window) - match.text_end
    else:
        raise ValueError(f"Unknown terminal: {terminal}")

    # No valid terminal-local match was found.
    if not matches:
        return None

    # Reject ambiguous matches when multiple barcode identities are present.
    if (
        require_unique_barcode
        and len({barcode_names[match.pattern_idx] for match in matches}) > 1
    ):
        return None

    # Prefer the lowest-edit match, then the one closest to the terminal.
    match = min(
        matches,
        key=lambda candidate: (candidate.cost, placement(candidate)),
    )

    # Convert the searcher's result into the application's match type.
    return TerminalMatch(
        barcode=barcode_names[match.pattern_idx],
        start=match.text_start,
        end=match.text_end,
        cost=match.cost,
        cigar=match.cigar,
    )


In [ ]:
def best_left_terminal_match(window: str) -> TerminalMatch | None:
    return (
        best_terminal_match(window, LEFT_CONSTRUCTS, "left")
        or best_terminal_match(window, LEFT_PARTIAL_ADAPTER_CONSTRUCTS, "left")
        or best_terminal_match(
            window,
            LEFT_PARTIAL_BARCODE_CONSTRUCTS,
            "left",
            LEFT_PARTIAL_TERMINAL_SLACK,
            LEFT_PARTIAL_MAX_EDITS,
            True,
        )
    )


def best_right_terminal_match(window: str) -> TerminalMatch | None:
    return (
        best_terminal_match(window, RIGHT_ADAPTER_CONSTRUCTS, "right")
        or best_terminal_match(window, RIGHT_BARCODE_CONSTRUCTS, "right")
        or best_terminal_match(
            window,
            RIGHT_PARTIAL_BARCODE_CONSTRUCTS,
            "right",
            RIGHT_PARTIAL_TERMINAL_SLACK,
            RIGHT_PARTIAL_MAX_EDITS,
            True,
        )
    )

In [ ]:
def trim_record(record: FastqRecord) -> TrimDecision:
    left_window = record.sequence[:TERMINAL_WINDOW]
    right_window_start = max(0, len(record.sequence) - TERMINAL_WINDOW)
    right_window = record.sequence[right_window_start:]

    left_match = best_left_terminal_match(left_window)
    right_match = best_right_terminal_match(right_window)

    left_cut = left_match.end if left_match else 0
    right_cut = right_window_start + right_match.start if right_match else len(record.sequence)
    if left_cut > right_cut:
        left_match = None
        right_match = None
        left_cut = 0
        right_cut = len(record.sequence)

    return TrimDecision(
        record=FastqRecord(
            title=record.title,
            sequence=record.sequence[left_cut:right_cut],
            qualities=record.qualities[left_cut:right_cut],
        ),
        left_match=left_match,
        right_match=right_match,
    )

## subsample test for debugging

The workflow builds 96 left constructs from the ligation-adapter top strand, native forward flanks, and each table forward barcode. It also builds a 52-base left fallback containing the final 12 adapter bases and the full flanked barcode, which recovers constructs with a truncated adapter after bounded leading noise. A final 32-base forward-barcode-plus-right-flank fallback permits five edits only within five bases of the left boundary, and only if every candidate reports the same barcode. It builds 96 full right constructs from the native reverse flanks, each table reverse barcode, and the ligation-adapter bottom strand; if the bottom strand is absent from a simplex read, it falls back to the 45-base reverse barcode construct. Its final 31-base reverse-flank-plus-barcode recovery path applies the same five-edit, immediate-boundary, unambiguous-barcode safeguards. Together these paths recover barcodes truncated at either read boundary without making interior or ambiguous barcode calls.

In [ ]:
sample_records = list(islice(read_fastq(INPUT_FASTQ), 250))
sample_decisions = [trim_record(record) for record in sample_records]

left_trimmed = sum(decision.left_match is not None for decision in sample_decisions)
right_trimmed = sum(decision.right_match is not None for decision in sample_decisions)
both_trimmed = sum(
    decision.left_match is not None and decision.right_match is not None
    for decision in sample_decisions
)
left_barcodes = Counter(
    decision.left_match.barcode for decision in sample_decisions if decision.left_match
)
right_barcodes = Counter(
    decision.right_match.barcode for decision in sample_decisions if decision.right_match
)
print(
    f"sample reads: {len(sample_records)}\n"
    f"left constructs removed: {left_trimmed}\n"
    f"right constructs removed: {right_trimmed}\n"
    f"both constructs removed: {both_trimmed}\n"
    f"left barcode calls: {left_barcodes.most_common(5)}\n"
    f"right barcode calls: {right_barcodes.most_common(5)}"
)

for original, decision in [
    (record, decision)
    for record, decision in zip(sample_records, sample_decisions)
    if decision.left_match or decision.right_match
][:10]:
    print(
        original.title.split()[0],
        f"{len(original.sequence)} -> {len(decision.record.sequence)}",
        f"left={decision.left_match}",
        f"right={decision.right_match}",
        sep=" | ",
    )


## Write cleaned reads

The full pass streams the compressed input and writes an uncompressed FASTQ. A read is modified only when a construct passes the terminal placement and edit-distance checks; every output record retains its original title and synchronized quality string.

In [ ]:
total_reads = 0
left_removed = 0
right_removed = 0
both_removed = 0
bases_removed = 0
left_barcodes = Counter()
right_barcodes = Counter()

with OUTPUT_FASTQ.open("w") as output_handle:
    for original in read_fastq(INPUT_FASTQ):
        decision = trim_record(original)
        cleaned = decision.record
        assert len(cleaned.sequence) == len(cleaned.qualities)
        assert len(cleaned.sequence) <= len(original.sequence)

        output_handle.write(
            f"@{cleaned.title}\n{cleaned.sequence}\n+\n{cleaned.qualities}\n"
        )
        total_reads += 1
        left_removed += decision.left_match is not None
        right_removed += decision.right_match is not None
        both_removed += (
            decision.left_match is not None and decision.right_match is not None
        )
        bases_removed += len(original.sequence) - len(cleaned.sequence)
        if decision.left_match:
            left_barcodes[decision.left_match.barcode] += 1
        if decision.right_match:
            right_barcodes[decision.right_match.barcode] += 1

print(
    f"input reads: {total_reads:,}\n"
    f"left constructs removed: {left_removed:,}\n"
    f"right constructs removed: {right_removed:,}\n"
    f"both constructs removed: {both_removed:,}\n"
    f"bases removed: {bases_removed:,}\n"
    f"left barcode calls: {left_barcodes.most_common()}\n"
    f"right barcode calls: {right_barcodes.most_common()}\n"
    f"output: {OUTPUT_FASTQ.relative_to(DIRECTORY_ROOT)}"
)

## Output validation

Read the cleaned FASTQ from disk and verify that every source record remains present in the same order, with matching sequence and quality lengths.

In [ ]:
validated_reads = 0
observed_removed_bases = 0

input_records = read_fastq(INPUT_FASTQ)
output_records = read_fastq(OUTPUT_FASTQ)
for source, cleaned in zip(input_records, output_records):
    assert cleaned.title == source.title
    assert len(cleaned.sequence) == len(cleaned.qualities)
    assert len(cleaned.sequence) <= len(source.sequence)
    observed_removed_bases += len(source.sequence) - len(cleaned.sequence)
    validated_reads += 1
assert next(input_records, None) is None
assert next(output_records, None) is None

assert validated_reads == total_reads
assert observed_removed_bases == bases_removed
assert OUTPUT_FASTQ.stat().st_size > 0
print(f"Validated {validated_reads:,} FASTQ records and {observed_removed_bases:,} removed bases.")